In [ ]:
!pip install -q google-genai python-dotenv pypdf numpy pandas

In [ ]:
import os
import json
import numpy as np

from google import genai

print("Libraries imported successfully")

Libraries imported successfully


In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets")

print("API key loaded successfully")

API key loaded successfully


In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client created successfully")

Gemini client created successfully


In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain Python in very simple words."
)

print(response.text)

Imagine you want to tell a robot how to make a peanut butter sandwich. 

Since the robot doesn't understand human speech, you need to write the instructions in a language it *does* understand. 

**Python is that language.**

Here is the simplest way to understand Python:

### 1. It is like talking in English
Some computer languages are full of strange symbols and look like secret codes. Python is different. It uses normal English words. 
* If you want the computer to write "Hello" on the screen, you just type: 
  `print("Hello")`

### 2. It is super friendly for beginners
Python is designed to be easy to read and write. Because it is so simple, you can learn the basics in just a few days. 

### 3. What can you do with Python?
Python is like a Swiss Army knife—you can use it for almost anything:
* **Build games** (like Minecraft addons)
* **Make websites** (Instagram and YouTube are built using Python!)
* **Create AI** (like robots that can talk or self-driving cars)
* **Do your chores*

In [ ]:
texts = [
    "Python is used for programming and data analysis.",
    "Python is a programming language used to analyze data.",
    "I enjoy playing cricket and watching movies."
]

print("Texts created successfully")

Texts created successfully


In [ ]:
embeddings = []

for text in texts:
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )

    embeddings.append(result.embeddings[0].values)

print("Embeddings created successfully")
print("Number of embeddings:", len(embeddings))
print("Embedding size:", len(embeddings[0]))

Embeddings created successfully
Number of embeddings: 3
Embedding size: 3072


In [ ]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

print("Cosine similarity function created")

Cosine similarity function created


In [ ]:
similar_score = cosine_similarity(
    embeddings[0],
    embeddings[1]
)

print("Similar sentence score:", similar_score)

Similar sentence score: 0.8607223293959226


In [ ]:
unrelated_score = cosine_similarity(
    embeddings[0],
    embeddings[2]
)

print("Unrelated sentence score:", unrelated_score)

Unrelated sentence score: 0.5503448555676841


In [ ]:
print("Similar score:", similar_score)
print("Unrelated score:", unrelated_score)

if similar_score > unrelated_score:
    print("SUCCESS: Similar meaning has a higher similarity score.")
else:
    print("Check the embeddings or model.")

Similar score: 0.8607223293959226
Unrelated score: 0.5503448555676841
SUCCESS: Similar meaning has a higher similarity score.


In [19]:
sample_resume = """
SINDHU JAVARI

Email: sindhujavari@gmail.com
Phone: +91 95351 84639

Education:
B.Tech in Computer Science Engineering (AI/ML)
Alliance University
2023 - 2027

Skills:
Python, SQL, Machine Learning, Deep Learning, NLP,
Computer Vision, Power BI, Tableau, HTML, CSS

Experience:
AI and Data Analyst Intern:
Analyzed structured datasets using Python and SQL.
Created dashboards using Power BI and Tableau.

Full Stack Developer Intern:
Built front-end and back-end components using HTML, CSS and Python.

Projects:
Cyber Threat Analysis Project:
Developed a machine learning system to identify potential cyber threats.

Optimized Crop Recommendation:
Built an IoT-based crop recommendation system using soil,
temperature and rainfall data.

Target Role:
AI/ML Engineer
"""

print(sample_resume)


SINDHU JAVARI

Email: sindhujavari@gmail.com
Phone: +91 95351 84639

Education:
B.Tech in Computer Science Engineering (AI/ML)
Alliance University
2023 - 2027

Skills:
Python, SQL, Machine Learning, Deep Learning, NLP,
Computer Vision, Power BI, Tableau, HTML, CSS

Experience:
AI and Data Analyst Intern:
Analyzed structured datasets using Python and SQL.
Created dashboards using Power BI and Tableau.

Full Stack Developer Intern:
Built front-end and back-end components using HTML, CSS and Python.

Projects:
Cyber Threat Analysis Project:
Developed a machine learning system to identify potential cyber threats.

Optimized Crop Recommendation:
Built an IoT-based crop recommendation system using soil,
temperature and rainfall data.

Target Role:
AI/ML Engineer



In [20]:
resume_prompt = f"""
You are an expert resume parser.

Extract information from the resume given below.

Return ONLY valid JSON.

The JSON must contain exactly these fields:

{{
    "name": "",
    "email": "",
    "phone": "",
    "skills": [],
    "experience": [],
    "education": [],
    "target_role": ""
}}

Rules:
1. Do not invent information.
2. Use an empty string if a field is not available.
3. Use an empty list if no information is available.
4. Skills must be a list.
5. Experience must be a list.
6. Education must be a list.
7. Target role should be a string.
8. Return ONLY JSON. Do not use markdown.

Resume:
{sample_resume}
"""

print("Resume parser prompt created successfully")

Resume parser prompt created successfully


In [24]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=resume_prompt
)

print(response.text)

{
    "name": "SINDHU JAVARI",
    "email": "sindhujavari@gmail.com",
    "phone": "+91 95351 84639",
    "skills": [
        "Python",
        "SQL",
        "Machine Learning",
        "Deep Learning",
        "NLP",
        "Computer Vision",
        "Power BI",
        "Tableau",
        "HTML",
        "CSS"
    ],
    "experience": [
        "AI and Data Analyst Intern: Analyzed structured datasets using Python and SQL. Created dashboards using Power BI and Tableau.",
        "Full Stack Developer Intern: Built front-end and back-end components using HTML, CSS and Python."
    ],
    "education": [
        "B.Tech in Computer Science Engineering (AI/ML), Alliance University, 2023 - 2027"
    ],
    "target_role": "AI/ML Engineer"
}


In [29]:
import json
import re

result_text = response.text.strip()

# Remove ```json and ``` if Gemini added them
result_text = re.sub(r"```json", "", result_text, flags=re.IGNORECASE)
result_text = result_text.replace("```", "").strip()

# Keep only the JSON object
start = result_text.find("{")
end = result_text.rfind("}")

if start == -1 or end == -1:
    raise ValueError("Gemini did not return valid JSON.")

result_text = result_text[start:end + 1]

print(result_text)

{
    "name": "SINDHU JAVARI",
    "email": "sindhujavari@gmail.com",
    "phone": "+91 95351 84639",
    "skills": [
        "Python",
        "SQL",
        "Machine Learning",
        "Deep Learning",
        "NLP",
        "Computer Vision",
        "Power BI",
        "Tableau",
        "HTML",
        "CSS"
    ],
    "experience": [
        "AI and Data Analyst Intern: Analyzed structured datasets using Python and SQL. Created dashboards using Power BI and Tableau.",
        "Full Stack Developer Intern: Built front-end and back-end components using HTML, CSS and Python."
    ],
    "education": [
        "B.Tech in Computer Science Engineering (AI/ML), Alliance University, 2023 - 2027"
    ],
    "target_role": "AI/ML Engineer"
}


In [30]:
profile = json.loads(result_text)

print("JSON converted successfully!")
print(json.dumps(profile, indent=2))

JSON converted successfully!
{
  "name": "SINDHU JAVARI",
  "email": "sindhujavari@gmail.com",
  "phone": "+91 95351 84639",
  "skills": [
    "Python",
    "SQL",
    "Machine Learning",
    "Deep Learning",
    "NLP",
    "Computer Vision",
    "Power BI",
    "Tableau",
    "HTML",
    "CSS"
  ],
  "experience": [
    "AI and Data Analyst Intern: Analyzed structured datasets using Python and SQL. Created dashboards using Power BI and Tableau.",
    "Full Stack Developer Intern: Built front-end and back-end components using HTML, CSS and Python."
  ],
  "education": [
    "B.Tech in Computer Science Engineering (AI/ML), Alliance University, 2023 - 2027"
  ],
  "target_role": "AI/ML Engineer"
}


In [31]:
required_fields = [
    "name",
    "email",
    "phone",
    "skills",
    "experience",
    "education",
    "target_role"
]

for field in required_fields:
    if field not in profile:
        raise ValueError(f"Missing field: {field}")

assert isinstance(profile["skills"], list)
assert isinstance(profile["experience"], list)
assert isinstance(profile["education"], list)
assert isinstance(profile["target_role"], str)

print("✅ Resume JSON validation successful!")

✅ Resume JSON validation successful!
